In [2]:
import os
from itertools import chain
from logging import debug

from dotenv import load_dotenv

load_dotenv()
model_name = os.getenv("LOCAL_MODE")
base_url = os.getenv("LOCAL_BASE_URL")

In [3]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(
    model=model_name,
    base_url=base_url,
    temperature=0.2
)

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 定义 prompt 模板
prompt = ChatPromptTemplate.from_template(
    "请用通俗易懂的语言解释一下什么是“{topic}”？"
)

# 2. 定义解析器
out_parser = StrOutputParser()

# 3. 使用管道符组装连接
chain = prompt | llm | out_parser

# 4. 调用链
question = {"topic":"人工智能"}

responses = chain.invoke(question)
print(responses)

😊

"人工智能" (Artificial Intelligence, AI) is a technology that allows computers to think and learn like humans. Just like how you can recognize objects, understand language, and make decisions based on experiences, AI enables machines to do the same.

Think of it like this: Imagine you're playing chess with a friend. You both move pieces around, trying to outsmart each other. A human player would use their knowledge, experience, and intuition to make moves. An AI system, on the other hand, uses algorithms (like a set of instructions) to analyze the game board, predict your next move, and respond accordingly.

AI is not just about playing chess or solving puzzles; it's about creating machines that can:

1. **Learn**: Just like how you learn from experiences, AI systems can absorb data, recognize patterns, and adapt to new situations.
2. **Reason**: AI can analyze information, draw conclusions, and make decisions based on that analysis.
3. **Perceive**: AI can understand and interpret sens

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain

prompt = PromptTemplate(
    input_variables=["input"],
    template="请用通俗的语言解释一下什么是：{topic}"
)

# 2. 使用 LLMChain 封装
# chain = LLMChain(llm=llm, prompt=prompt)
chain = prompt | llm

# 3. 调用链
question = {"topic": "量子计算"}
response = chain.invoke(question)

print(response)

A great topic! 😄

Quantum computing is a new way of processing information that's different from the classical computers we're familiar with. Let me break it down in simple terms:

**Classical Computing**

In traditional computers, information is stored and processed using "bits" (0s and 1s). These bits are like light switches - they can be either ON (1) or OFF (0), but not both at the same time.

Think of a bit like a coin: it's either heads or tails, but never both simultaneously. This is known as a "classical" or "binary" system.

**Quantum Computing**

Now, imagine a special kind of coin that can exist in multiple states at once - like being both heads and tails simultaneously! 🔮 This is the essence of quantum computing.

In a quantum computer, information is stored and processed using "qubits" (quantum bits). Qubits are like these special coins that can exist in many states at the same time. This means they can process multiple possibilities simultaneously, which allows for much f

In [10]:
from langchain_core.runnables import RunnableParallel
# 定义一个用于生成摘要的链
summarize_chain = (
    ChatPromptTemplate.from_template("请为以下内容生成一段摘要：\n\n{context}")
    | llm
    | StrOutputParser()
)

# 定义一个用于提取关键词的链
keywords_chain = (
    ChatPromptTemplate.from_template("请从以下内容中提取3个关键词：\n\n{context}")
    | llm
    | StrOutputParser()
)

# 使用 RunnableParallel 并行运行两个链
# 输入的 'context' 会同时传递给 summarize_chain 和 keywords_chain
parallel_chain = RunnableParallel(
    summary=summarize_chain,
    keywords=keywords_chain
)

# 调用并行链
context_text = "LangChain 是一个强大的框架，用于构建由大型语言模型驱动的应用程序。它提供了模块化的组件和链式调用的能力。"
result = parallel_chain.invoke({"context": context_text})

# print(result)

print(f"摘要: {result['summary']}")
print(f"关键词: {result['keywords']}")

{'summary': 'Here is a summary of the content:\n\nLangChain is a powerful framework for building applications driven by large language models. It offers modular components and chainable call abilities, enabling developers to create complex systems with ease.', 'keywords': 'Based on the content, I extracted three key words:\n\n1. LangChain\n2. Framework\n3. Language'}
摘要: Here is a summary of the content:

LangChain is a powerful framework for building applications driven by large language models. It offers modular components and chainable call abilities, enabling developers to create complex systems with ease.
关键词: Based on the content, I extracted three key words:

1. LangChain
2. Framework
3. Language


In [11]:
from langchain_core.prompts import ChatPromptTemplate
# 1. 定义 prompt 模板
prompt = ChatPromptTemplate.from_template(
    "请用通俗的语言解释一下什么是 '{topic}' "
)
# 2. 定义输出解析器
out_parser = StrOutputParser()

# 3. 使用管道符
chain = prompt | llm | out_parser

#4. 调用 链。使用 .stream() 方法进行流式调用
stream = chain.stream({"topic":"黑洞"})
print("模型正在流式输出：")
for chunk in stream:
    # chunk 是一个字符串片段
    print(chunk, end="", flush=True)

print("\n\n流式输出结束。")

模型正在流式输出：
What a great question! 😄

So, you know how stars are like big balls of hot, glowing gas in space? Well, when a star runs out of fuel and dies, it can sometimes collapse under its own gravity and shrink down to a tiny point called a singularity. This is kind of like what happens when you squish up a bunch of playdough into a tiny ball - the gravity gets so strong that nothing, not even light, can escape once it gets too close.

This tiny point is surrounded by an event horizon, which is like a boundary beyond which anything that crosses it gets pulled in and trapped forever. Think of it like a cosmic sinkhole! 🌊 Once something crosses the event horizon, it's gone for good - no signals, no messages, nothing can escape to tell us what happened.

Now, here's where things get really weird: because gravity is so strong near the singularity, time and space become all mixed up. Imagine you're standing next to a super-powerful magnet, and your watch starts running slower than usual be

In [12]:
from langchain_core.runnables import RunnableLambda
def log_and_pass(x):
    """一个简单的调试函数，打印输入并原样返回"""
    print("--- 调试信息 ---")
    print(f"当前步骤的输入: {x}")
    print("------------------")
    return x

# 在 prompt 和 llm 之间插入调试节点
debug_chain = prompt | RunnableLambda(log_and_pass) | llm

debug_chain.invoke({"topic": "人工智能"})

--- 调试信息 ---
当前步骤的输入: messages=[HumanMessage(content="请用通俗的语言解释一下什么是 '人工智能' ", additional_kwargs={}, response_metadata={})]
------------------


'😊\n\n"人工智能" (píng rén gōu zhì) is a term that refers to the ability of machines, like computers and robots, to think and learn like humans. In other words, it\'s when machines can make decisions, solve problems, and even understand us in a way that\'s similar to how we humans do.\n\nThink about it like this: you\'re playing chess with a friend, and they\'re really good at it. They can anticipate your moves, adjust their strategy, and win the game. That\'s basically what artificial intelligence (AI) is – machines that can "think" ahead of time, make smart decisions, and adapt to new situations.\n\nThere are many types of AI, but some common examples include:\n\n1. **Chatbots**: These are computer programs that can have conversations with humans, like Siri or Alexa.\n2. **Image recognition**: AI systems that can identify objects in pictures or videos, like facial recognition software.\n3. **Natural Language Processing (NLP)**: AI that can understand and generate human language, like tex

In [13]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

# 步骤1：定义 Pydantic 模型用于结构化输出
class Entity(BaseModel):
    name: str = Field(description="需要查询的实体名称")
    topic: str = Field(description="实体所属的主题领域")

# 步骤1：创建解析器和提示
json_parser = JsonOutputParser(pydantic_object=Entity)
prompt1 = ChatPromptTemplate.from_template(
    "从用户的问题中提取出核心实体。\n{format_instructions}\n用户问题：{question}"
).partial(format_instructions=json_parser.get_format_instructions())

# 步骤1：构建第一条链
chain1 = prompt1 | llm | json_parser

# 步骤2：定义第二条链的提示
prompt2 = ChatPromptTemplate.from_template(
    "请详细解释一下在“{topic}”领域中的“{name}”是什么。"
)

# 步骤2：构建第二条链
chain2 = prompt2 | llm | StrOutputParser()

# 完整的多步骤链
# chain1 的输出（一个字典）会自动作为 chain2 的输入
multi_step_chain = chain1 | chain2

# 调用整个链
user_question = "我想知道在计算机科学里，什么是图灵机？"
final_response = multi_step_chain.invoke({"question": user_question})

print(final_response)

A classic topic in computer science! 😊

In the field of computer science, a Turing machine is a mathematical model for computation that was introduced by Alan Turing in his 1936 paper "On Computable Numbers." It's a simple, abstract device that can perform computations by reading and writing symbols on an infinite tape.

**Basic Components:**

1. **Tape:** An infinite, one-dimensional tape divided into cells, each of which can hold a symbol from a finite alphabet (e.g., 0s and 1s).
2. **Head:** A read/write device that can move along the tape, reading and writing symbols.
3. **State:** The machine's current state, which determines its behavior.

**Operations:**

1. **Read:** The head reads the symbol on the current cell of the tape.
2. **Write:** The head writes a new symbol on the current cell of the tape.
3. **Move:** The head moves one cell to the left or right along the tape.
4. **Change State:** The machine changes its state based on the read symbol and its current state.

**Turin